In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

In [2]:
MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "DataL@b_Cae123"
NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1" 

conf = (
    SparkConf()
    .setAppName("Job_Transformation_silver")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.extraJavaOptions", "-Dorg.apache.poi.util.IOUtils.setByteArrayMaxOverride=1000000000")
    .set("spark.driver.memory", "16g")
    # AJOUT DES PACKAGES : On ajoute le jar nessie-spark-extensions
    .set("spark.jars.packages", 
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,""com.crealytics:spark-excel_2.12:3.5.0_0.20.3")
     
     .set("spark.sql.extensions", 
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    
    # On définit le bucket bronze comme entrepôt par défaut du catalogue
    .set("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    
    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [3]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

/usr/local/lib/python3.12/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
com.crealytics#spark-excel_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d344a67a-af19-451e-aa14-575b2d37be42;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
	found com.crealytics#spark-excel_2.12;3.5.0_0.20.3 in central
	found org.apache.poi#poi;5.2.5 in central
	found commons-codec#commons-codec;1.

In [ ]:
# Créer la branche de transformation depuis main
spark.sql("CREATE BRANCH IF NOT EXISTS `transform-silver` IN nessie FROM main")
spark.sql("USE REFERENCE `transform-silver` IN nessie")


DataFrame[refType: string, name: string, hash: string]

In [5]:
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

+-------+----------------+----------------------------------------------------------------+
|refType|name            |hash                                                            |
+-------+----------------+----------------------------------------------------------------+
|Branch |transform-silver|296763eacd1d68ae8248acb10fb240be67e937e38a7cc9c40037667fec0bc42b|
+-------+----------------+----------------------------------------------------------------+



In [ ]:
# Lire depuis bronze sur la branche courante
df_bronze = spark.sql("SELECT * FROM nessie.bronze.extrait_solde")

26/04/13 16:12:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


## ON EFFECTUE ICI LES TRANSFORMATIONS NECESSAIRES

In [ ]:
print(f"Nombre de lignes bronze : {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)


Nombre de lignes bronze : 225398
root
 |-- SERVICE: string (nullable = true)
 |-- ORGANISME: string (nullable = true)
 |-- EMPLOI: string (nullable = true)
 |-- GRADE: string (nullable = true)
 |-- CLASSE/ECHELON: string (nullable = true)



[Stage 3:>                                                          (0 + 1) / 1]

+-----------------------------------------+------------------------------------------------------------------------------------+-----------------+-------------------------------------------------+------------------------+
|SERVICE                                  |ORGANISME                                                                           |EMPLOI           |GRADE                                            |CLASSE/ECHELON          |
+-----------------------------------------+------------------------------------------------------------------------------------+-----------------+-------------------------------------------------+------------------------+
|O N I                                    |Min. d'Etat Administration du Territoire                                            |NULL             |Non Fonctionnaire Personnel à Décision Provisoire|NULL                    |
|Ecole Normale Supérieure                 |Ecole Normale Supérieure                                             

In [ ]:
# Légère transformation
df_silver = df_bronze \
    .dropDuplicates() \
    .dropna()

In [11]:
print(f"Nombre de lignes silver : {df_silver.count()}")
df_silver.show(5, truncate=False)

Nombre de lignes silver : 34715
+---------------------------+------------------------------------------------------+----------------------------------------+----------------------+----------------------------------+
|SERVICE                    |ORGANISME                                             |EMPLOI                                  |GRADE                 |CLASSE/ECHELON                    |
+---------------------------+------------------------------------------------------+----------------------------------------+----------------------+----------------------------------+
|D P I F R                  |Min. des Eaux et Forêts                               |Ingénieur des Techniques: Eaux et Forêts|Fonctionnaire Grade A3|Classe Exceptionnelle 3ème Echelon|
|E P H D                    |Min. de la Santé, de l'Hygiène Publique et de la C M U|Infirmier Diplomé d'Etat                |Fonctionnaire Grade B3|Classe Exceptionnelle 3ème Echelon|
|Lycée Moderne Harris Adjamé|Min. de l'Education

## ON CREE LA BASE DANS LA COUCHE CORRESPONDANTE

In [8]:
# Créer namespace silver
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

DataFrame[]

In [12]:
# Écrire dans silver
df_silver.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("nessie.silver.extrait_solde")

Transformation silver terminée !


In [14]:
# Merger dans main
spark.sql("MERGE BRANCH `transform-silver` INTO main IN nessie")

Merge dans main terminé !


In [ ]:
# On supprime la branche 
spark.sql("DROP BRANCH `transform-silver` IN nessie")